In [ ]:
import sys
from pathlib import Path

# Make src/ importable when running from notebooks/
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

CORPUS_PATH = REPO_ROOT / "data" / "corpus.parquet"
corpus = pd.read_parquet(CORPUS_PATH, engine="pyarrow")

print("Shape  :", corpus.shape)
print("Columns:", list(corpus.columns))
print()
print("Dtypes:")
print(corpus.dtypes.to_string())
print()
print("Null counts:")
print(corpus.isnull().sum().to_string())

In [ ]:
counts = (
    corpus.groupby(["dataset", "task_type"])["id"]
    .count()
    .rename("examples")
    .reset_index()
    .sort_values("examples", ascending=False)
)

print(f"{'dataset':<22} {'task_type':<18} {'examples':>10}")
print("-" * 55)
for _, row in counts.iterrows():
    print(f"{row['dataset']:<22} {row['task_type']:<18} {row['examples']:>10,}")
print("-" * 55)
print(f"{'TOTAL':<22} {'':18} {counts['examples'].sum():>10,}")

In [ ]:
import textwrap

task_types = ["math_reasoning", "commonsense", "nli", "mcqa", "extractive_qa"]

for tt in task_types:
    row = corpus[corpus["task_type"] == tt].iloc[0]
    print(f"=== {tt} ===")
    print(f"  dataset  : {row['dataset']}")
    print(f"  language : {row['language']}")
    print(f"  id       : {row['id']}")
    print(f"  input    : {textwrap.shorten(str(row['input']), 120)}")
    print(f"  choices  : {str(row['choices'])[:80]}")
    print(f"  label    : {row['label']}")
    print()

In [ ]:
from src.data.preprocessor import format_example

task_types = ["math_reasoning", "commonsense", "nli", "mcqa", "extractive_qa"]

for tt in task_types:
    row = corpus[corpus["task_type"] == tt].iloc[0]
    prompt = format_example(row)
    print(f"{'=' * 70}")
    print(f"task_type : {tt}  |  dataset : {row['dataset']}  |  lang : {row['language']}")
    print(f"{'=' * 70}")
    print(prompt[:600])
    if len(prompt) > 600:
        print("  ... [truncated]")
    print()

In [ ]:
import matplotlib.pyplot as plt

ds_counts = (
    corpus.groupby("dataset")["id"]
    .count()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(ds_counts.index, ds_counts.values, color="steelblue", edgecolor="white")

# Annotate each bar with the count
for bar, val in zip(bars, ds_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 300,
        f"{val:,}",
        ha="center", va="bottom", fontsize=9,
    )

ax.set_title("Example counts per dataset", fontsize=13, pad=12)
ax.set_xlabel("Dataset")
ax.set_ylabel("Number of examples")
ax.set_xticks(range(len(ds_counts)))
ax.set_xticklabels(ds_counts.index, rotation=30, ha="right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()